In [24]:
"""
Импорт необходимых библиотек.
pandas - для работы с табличными данными.
numpy - для работы с массивами.
scipy.sparse - для создания разреженных матриц CSR.
implicit - библиотека для работы с коллаборативной фильтрацией (ALS и др.).
faiss - для поиска по векторным представлениям.
sentence_transformers - для генерации текстовых эмбеддингов.
openai - клиент для работы с API (совместимость с OpenRouter).
tqdm - для индикации прогресса.
os, dotenv - для конфигурации переменных окружения.
pickle - для сохранения и загрузки объектов (маппингов и моделей).
"""
import pandas as pd
import numpy as np
import scipy.sparse
import implicit
import faiss
from sentence_transformers import SentenceTransformer
import openai
from tqdm import tqdm
import os
import pickle
from dotenv import load_dotenv

tqdm.pandas()

"""
Загрузка переменных окружения.
"""
load_dotenv()

True

In [25]:
"""
Загрузка данных.
Используем заранее очищенные данные фильмов (movies_cleaned.csv) и исходные рейтинги (ratings_small.csv).
Обеспечиваем одинаковый тип данных для идентификаторов (movieId и id).
"""
movies = pd.read_csv('data/movies_cleaned.csv')
ratings = pd.read_csv('data/ratings_small.csv', usecols=['userId', 'movieId', 'rating'])

movies['id'] = movies['id'].fillna(0).astype(int)
ratings['movieId'] = ratings['movieId'].fillna(0).astype(int)

"""
Оставляем в рейтингах только те фильмы, которые есть в базе фильмов.
"""
ratings = ratings[ratings['movieId'].isin(movies['id'])]

In [26]:
"""
Создание маппингов (индексов) для пользователей и фильмов.
Необходимо для создания разреженной матрицы и работы с алгоритмом ALS.
user_map (userId -> index), user_inv_map (index -> userId)
movie_map (movieId -> index), movie_inv_map (index -> movieId)
"""
user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

user_map = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_map = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

user_inv_map = {idx: user_id for user_id, idx in user_map.items()}
movie_inv_map = {idx: movie_id for movie_id, idx in movie_map.items()}

ratings['user_idx'] = ratings['userId'].map(user_map)
ratings['movie_idx'] = ratings['movieId'].map(movie_map)

print(f"Пользователей: {len(user_map)}, Фильмов: {len(movie_map)}")

"""
Сохранение маппингов на диск.
В реальном проекте это критически важно для согласованности данных между обучением и inference.
"""
with open('data/user_map.pkl', 'wb') as f:
    pickle.dump(user_map, f)
with open('data/movie_map.pkl', 'wb') as f:
    pickle.dump(movie_map, f)
with open('data/user_inv_map.pkl', 'wb') as f:
    pickle.dump(user_inv_map, f)
with open('data/movie_inv_map.pkl', 'wb') as f:
    pickle.dump(movie_inv_map, f)

Пользователей: 671, Фильмов: 2830


In [27]:
"""
Создание разреженной item-user матрицы.
Используется scipy.sparse.csr_matrix.
"""
rows = ratings['movie_idx'].values
cols = ratings['user_idx'].values
values = ratings['rating'].values

item_user_matrix = scipy.sparse.csr_matrix((values, (rows, cols)), shape=(len(movie_map), len(user_map)))
item_user_matrix.eliminate_zeros()

"""
Инициализация и обучение модели ALS.
factors=50, iterations=15, regularization=0.01.
"""
als_model = implicit.als.AlternatingLeastSquares(factors=75, regularization=0.01, iterations=15, random_state=42)
als_model.fit(item_user_matrix)

"""
Сохранение модели на диск.
В реальном проекте модель сохраняется для использования в сервисе рекомендаций.
"""
with open('data/als_model.pkl', 'wb') as f:
    pickle.dump(als_model, f)

  0%|          | 0/15 [00:00<?, ?it/s]

In [28]:
"""
Загрузка индекса FAISS и модели SentenceTransformer.
FAISS индекс загружается из файла data/movies_faiss.index.
Используется модель 'all-MiniLM-L6-v2' для векторизации текстовых запросов.
В реальном проекте индекс строится отдельно и подгружается на этапе inference.
"""
if os.path.exists('data/movies_faiss.index'):
    faiss_index = faiss.read_index('data/movies_faiss.index')
    print("FAISS индекс успешно загружен.")
else:
    print("Ошибка: FAISS индекс не найден. Сначала выполните блокнот 02_content_embeddings.ipynb.")

sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

FAISS индекс успешно загружен.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
"""
Функция поиска кандидатов с помощью ALS.
Принимает user_id и количество top_k.
Возвращает список словарей, содержащих название ('title') и описание ('overview').
"""
def get_als_candidates(user_id, top_k=10):
    if user_id not in user_map:
        return []
    
    user_idx = user_map[user_id]
    user_items = item_user_matrix.T.tocsr() # users x items
    
    recommendations = als_model.recommend(user_idx, user_items[user_idx], N=top_k)
    
    if isinstance(recommendations, tuple):
        movie_indices = recommendations[0]
    elif len(recommendations) > 0 and isinstance(recommendations[0], (list, tuple, np.ndarray)):
        movie_indices = [r[0] for r in recommendations]
    else:
        movie_indices = recommendations
        
    candidates = []
    for idx in movie_indices:
        movie_id = movie_inv_map.get(idx)
        if movie_id:
            row = movies[movies['id'] == movie_id]
            if not row.empty:
                candidates.append({
                    'title': row.iloc[0]['title'],
                    'overview': row.iloc[0]['overview']
                })
    return candidates

In [30]:
"""
Функция поиска кандидатов с помощью FAISS (контентная фильтрация).
Преобразует текстовый запрос в вектор, нормализует и ищет ближайших соседей в индексе.
Возвращает список словарей ('title', 'overview').
"""
def get_faiss_candidates(query_text, top_k=10):
    query_vector = sentence_model.encode([query_text])
    query_vector = np.array(query_vector).astype('float32')
    faiss.normalize_L2(query_vector)
    
    distances, indices = faiss_index.search(query_vector, top_k)
    
    candidates = []
    # indices[0] - массив индексов ближайших соседей
    for idx in indices[0]:
        if 0 <= idx < len(movies):
            row = movies.iloc[idx]
            candidates.append({
                'title': row['title'],
                'overview': row['overview']
            })
            
    return candidates

In [31]:
"""
Функция для получения гибридных рекомендаций.
Объединяет кандидатов от ALS и FAISS.
Удаляет дубликаты по названию фильма, сохраняя уникальные записи.
Возвращает список уникальных словарей с фильмами.
"""
def get_hybrid_recommendations(user_id, query_text):
    als_candidates = get_als_candidates(user_id)
    faiss_candidates = get_faiss_candidates(query_text)
    
    unique_movies = {}
    
    # Добавляем кандидатов от ALS
    for movie in als_candidates:
        title = movie['title']
        if title not in unique_movies:
            unique_movies[title] = movie
            
    # Добавляем кандидатов от FAISS
    for movie in faiss_candidates:
        title = movie['title']
        if title not in unique_movies:
            unique_movies[title] = movie
            
    return list(unique_movies.values())

In [32]:
"""
Генерация ответа с помощью LLM (OpenAI API через OpenRouter).
Функция generate_llm_response принимает запрос пользователя, кандидатов и историю просмотров.
Формирует контекст для модели nvidia/nemotron-3-super-120b-a12b:free.
Модель выступает в роли интеллектуального кино-ассистента.
Возвращает текст рекомендации.
"""
def generate_llm_response(user_query, hybrid_candidates, user_top_movies):
    """
    Инициализация клиента OpenAI для работы с OpenRouter.
    API ключ берется из переменной окружения OPENROUTER_API_KEY.
    Базовый URL настроен на https://openrouter.ai/api/v1.
    """
    client = openai.OpenAI(
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1"
    )
    
    system_prompt = "Ты — интеллектуальный кино-ассистент. Твоя задача — выбрать 3 фильма из предложенного списка кандидатов и обосновать их, опираясь на запрос пользователя и его историю оценок."
    
    candidates_text = "\n".join([f"- {c['title']}: {c['overview']}" for c in hybrid_candidates])
    history_text = ", ".join(user_top_movies)
    
    """
    Формирование пользовательского промпта.
    Передаем три ключевые переменные:
    1. Запрос пользователя (user_query).
    2. Историю просмотров (history_text).
    3. Список кандидатов с описаниями (candidates_text).
    """
    user_prompt = f"""
    Запрос пользователя: "{user_query}"
    
    Любимые фильмы пользователя (история): {history_text}
    
    Список кандидатов (найдены системой рекомендаций):
    {candidates_text}
    
    Пожалуйста, выбери 3 наиболее подходящих фильма из списка кандидатов и объясни, почему они подходят этому пользователю. Не придумывай фильмы, используй только кандидатов.
    """
    
    try:
        response = client.chat.completions.create(
            model="nvidia/nemotron-3-super-120b-a12b:free",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Произошла ошибка при обращении к API: {e}"

In [33]:
"""
Тестовый блок.
Выбираем пользователя для теста (ID=1).
Формируем текстовый запрос.
Запускаем гибридную рекомендацию.
Пробуем обратиться к LLM (оборачиваем в try/except).
"""
user_id = 1
query = "Хочу посмотреть напряженный триллер с неожиданной концовкой"

# 1. Получение топ-3 фильмов из истории оценок пользователя
user_movies = ratings[ratings['userId'] == user_id]
top_rated = user_movies.sort_values(by='rating', ascending=False).head(3)
top_movies_titles = []

for _, row in top_rated.iterrows():
    m_id = row['movieId']
    m_row = movies[movies['id'] == m_id]
    if not m_row.empty:
        top_movies_titles.append(m_row.iloc[0]['title'])

print(f"Пользователь: {user_id}")
print(f"Любимые фильмы: {', '.join(top_movies_titles)}")
print(f"Запрос: {query}")

# 2. Получение гибридных кандидатов
hybrid_candidates = get_hybrid_recommendations(user_id, query)

print(f"\nНайдено кандидатов: {len(hybrid_candidates)}")
for i, candy in enumerate(hybrid_candidates, 1):
    print(f"{i}. {candy['title']}")

# 3. Вызов LLM
print("\n--- Генерация ответа от LLM ---")
try:
    final_response = generate_llm_response(query, hybrid_candidates, top_movies_titles)
    print(final_response)
except Exception as e:
    print(f"Ошибка при вызове LLM (так как ключ фиктивный, это нормально): {e}")

Пользователь: 1
Любимые фильмы: American Pie, Rocky III, Confidentially Yours
Запрос: Хочу посмотреть напряженный триллер с неожиданной концовкой

Найдено кандидатов: 20
1. Rebecca
2. Mere Brother Ki Dulhan
3. Joshua
4. Reclaim Your Brain
5. The Hunchback of Notre Dame
6. The Wrong Trousers
7. Roustabout
8. One on Top of the Other
9. Read It and Weep
10. The Legend of Bagger Vance
11. Patrioticheskaya Komediya
12. Nasha Russia: Yaytsa sudby
13. Dura
14. Ivan the Terrible, Part I
15. The Second Circle
16. Don't call him "Dimon"
17. 1612: Chronicles of the Dark Time
18. Vdrebezgi
19. An Example of Intonation
20. Pro Lyuboff

--- Генерация ответа от LLM ---
**Выбранные фильмы**

1. **Rebecca**  
2. **Joshua**  3. **One on Top of the Other**  

**Почему они подходят запросу «напряженный триллер с неожиданной концовкой» и истории оценок пользователя**

| Фильм | Как соответствует запросу | Связь с историей оценок |
|-------|---------------------------|--------------------------|
| **Rebecca